## Task 1 — topic categorization

Gemini tags each transcript with a topic from a fixed list. I went with an LLM over
keyword rules because it handles phrasing we didn't anticipate, and the Pydantic
schema keeps the labels constrained to the taxonomy.

Data is 99 synthetic transcripts (the sample wasn't shared so I generated them —
note at the bottom). Needs `GEMINI_API_KEY` in `.env`.

In [ ]:
import pandas as pd

DATA = "data"

def load_transcripts():
    """Load every transcript + its metadata into one DataFrame (full text + customer-only voice)."""
    meta = pd.read_csv(f"{DATA}/metadata.csv")
    texts, voices = [], []
    for _, row in meta.iterrows():
        raw = open(f"{DATA}/transcripts/{row.call_type}/{row.call_id}.txt").read()
        body = raw.split("-" * 40, 1)[-1].strip()          # skip the header block
        turns = [ln.split(":", 1) for ln in body.splitlines() if ":" in ln]
        texts.append(body)
        # for support/external we only care about the customer; internal is all-hands
        voices.append(" ".join(
            t.strip() for spk, t in turns
            if row.call_type == "internal" or spk.strip().lower().startswith("customer")))
    return meta.assign(text=texts, voice_text=voices)

df = load_transcripts()
print(len(df), "transcripts", dict(df.call_type.value_counts()))
df[["call_id", "call_type", "date", "topic"]].head()

99 transcripts {'external': np.int64(33), 'internal': np.int64(33), 'support': np.int64(33)}


,call_id,call_type,date,topic
0,EXT-001,external,2026-02-09,Renewal Negotiation
1,EXT-002,external,2026-02-28,Quarterly Business Review
2,EXT-003,external,2026-01-19,Adoption & Onboarding
3,EXT-004,external,2026-03-14,Churn Risk & Escalation
4,EXT-005,external,2026-01-25,Renewal Negotiation


In [ ]:
import os, enum
from google import genai
from google.genai import types
from pydantic import BaseModel

def load_env(path=".env"):
    """Read KEY=VALUE lines from a .env file into environment variables."""
    for line in open(path):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
load_env()

MODEL = "gemini-3.5-flash"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def ask(prompt, schema):
    """Send one prompt to Gemini and return the reply parsed into the given Pydantic schema."""
    resp = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
            # small thinking budget: a little reasoning for ambiguous calls without
            # eating much of the token budget (0 = off, -1 = dynamic/unbounded)
            thinking_config=types.ThinkingConfig(thinking_budget=512),
            max_output_tokens=65536))
    return resp.parsed

def run_batch(items, schema, instruction):
    """Send all (id, text) items to Gemini in one call and return {id: parsed record}."""
    block = "\n\n".join(f"### ID: {cid}\n{txt}" for cid, txt in items)
    return {r.id: r for r in ask(f"{instruction}\nItems:\n{block}", list[schema])}

In [ ]:
class Category(str, enum.Enum):
    BILLING     = "Billing & Pricing"
    AUTH        = "Authentication & Access"
    API         = "API & Integrations"
    RELIABILITY = "Reliability & Performance"
    DATA        = "Data & Reporting"
    ADOPTION    = "Adoption & Enablement"
    RENEWAL     = "Renewal & Commercial"
    PRODUCT     = "Product Feedback & Roadmap"

class Classification(BaseModel):
    id: str
    category: Category
    reason: str

CLASSIFY_PROMPT = (
    "Label each B2B SaaS call transcript by topic. For every item give the best "
    "category and a short reason (<=12 words). Keep the ID.")

def classify(df):
    """Tag each transcript with a topic category via Gemini and merge the labels onto df."""
    recs = run_batch(list(zip(df.call_id, df.text)), Classification, CLASSIFY_PROMPT)
    tagged = pd.DataFrame([{"call_id": r.id, "category": r.category.value,
                            "category_reason": r.reason[:80]} for r in recs.values()])
    return df.merge(tagged, on="call_id", how="left")

Classify everything in one call, then save the whole labeled dataset. Notebooks 2 and 3 just read this file — the transcripts are loaded and the LLM runs only once, here.

In [ ]:
from pathlib import Path
df = classify(df)
Path("output").mkdir(exist_ok=True)
df.to_csv("output/labeled.csv", index=False)   # transcripts + metadata + topic labels
print(df.category.value_counts().to_string())
df[["call_id", "call_type", "category", "category_reason"]].head(8)

category
Reliability & Performance     24
Product Feedback & Roadmap    22
Adoption & Enablement         11
Authentication & Access       11
API & Integrations            10
Data & Reporting               9
Renewal & Commercial           6
Billing & Pricing              6


,call_id,call_type,category,category_reason
0,EXT-001,external,Renewal & Commercial,Discussing renewal quote being higher than bud...
1,EXT-002,external,Data & Reporting,"Discussing measurable impact, outcomes, metric..."
2,EXT-003,external,Adoption & Enablement,Discussing low regular user login and onboardi...
3,EXT-004,external,Product Feedback & Roadmap,Discussing compliance features and compliance ...
4,EXT-005,external,Renewal & Commercial,Addressing a renewal quote hike due to organic...
5,EXT-006,external,Adoption & Enablement,Assisting new hires struggling with onboarding...
6,EXT-007,external,Data & Reporting,Reviewing quarterly metrics scorecard and comp...
7,EXT-008,external,Product Feedback & Roadmap,Gathering usability feedback on reporting lear...


One example per topic, with the model's reason — a quick read that the labels make sense.

In [ ]:
for cat in df.category.value_counts().index:
    ex = df[df.category == cat].iloc[0]
    print(f"{cat} ({ex.call_id}, {ex.call_type}): {ex.category_reason}")
    print("  ", " ".join(ex.text.split()[:22]), "...")

Reliability & Performance (EXT-014, external): Addressing customer's shaken confidence due to last quarter's outages.
   -------------------- AM (Priya): Hi Owen, thanks for making time today. How are things going on your side? Customer (Owen): The outages last ...
Product Feedback & Roadmap (EXT-004, external): Discussing compliance features and compliance roadmap commitment.
   -------------------- AM (Raj): Hi Yuki, thanks for making time today. How are things going on your side? Customer (Yuki): If we can't ...
Adoption & Enablement (EXT-003, external): Discussing low regular user login and onboarding new hires.
   -------------------- AM (Elena): Hi Carlos, thanks for making time today. How are things going on your side? Customer (Carlos): Only a handful ...
Authentication & Access (INT-003, internal): Sizing SSO reliability work and mapping implementation tickets.
   -------------------- Lead (Fatima): We need to size the SSO reliability work before we commit. Eng (Devin): Let's 

Dataset note: the original sample wasn't provided, so I generated 99 transcripts
(33 per type) with a seeded script. A few patterns are baked in on purpose
(billing/outage calls skew negative, internal incident calls echo support themes)
so there's something real for the sentiment/insight work to find.